In [1]:
"""
07_rl_count_refine.py

FruitNet experiment 07: Reinforcement Learning based counting refinement.
This script is designed to run immediately after 06_train.py / 06_train.ipynb.

Main outputs:
  - candidate prediction cache for train/val/test
  - PPO/contextual-bandit RL policy checkpoints: latest + best
  - fixed threshold, Soft-NMS, grid-search, and RL comparison tables
  - per-image predictions for manuscript analysis
  - detailed logs and resumable progress files

Run from project root:
  conda activate fruitnet
  cd /home/diy-hus/fruitnet-chili-yield
  python 07_rl_count_refine.py

Common overrides:
  export DATA_DIR=/mnt/f/fruitnet-chili-yield-data/finetune_data
  export OUTPUT_ROOT=/mnt/f/fruitnet-chili-yield-outputs
  export DETECTOR_KEY=fruitnet_gcs
  export DETECTOR_PATH=/mnt/f/fruitnet-chili-yield-outputs/artifacts/finetuned/fruitnet_gcs_finetune_data_best.pt
  export RL_EPOCHS=300
  export RL_TRAIN_SPLITS=val
  export FINAL_TEST_SPLIT=test
  python 07_rl_count_refine.py
"""

from __future__ import annotations

from pathlib import Path
from dataclasses import dataclass, asdict
from datetime import datetime
import os
import sys
import json
import time
import math
import random
import shutil
import logging
import platform
import traceback
from typing import Any, Dict, Iterable, List, Optional, Sequence, Tuple

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

try:
    import yaml
except Exception as e:  # pragma: no cover
    raise RuntimeError("Missing pyyaml. Install with: pip install pyyaml") from e

try:
    from PIL import Image
except Exception as e:  # pragma: no cover
    raise RuntimeError("Missing pillow. Install with: pip install pillow") from e

try:
    import torch
    import torch.nn as nn
    import torch.nn.functional as F
except Exception as e:  # pragma: no cover
    raise RuntimeError("Missing torch. Install PyTorch in the fruitnet conda environment.") from e


# ============================================================
# 0. Helpers: paths, logging, atomic writes
# ============================================================

IMG_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp", ".tif", ".tiff"}


def env_bool(name: str, default: bool) -> bool:
    raw = os.environ.get(name, "").strip().lower()
    if raw == "":
        return default
    return raw in {"1", "true", "yes", "y", "on"}


def env_int(name: str, default: int) -> int:
    raw = os.environ.get(name, "").strip()
    return int(raw) if raw else default


def env_float(name: str, default: float) -> float:
    raw = os.environ.get(name, "").strip()
    return float(raw) if raw else default


def parse_float_list(raw: str, default: Sequence[float]) -> List[float]:
    raw = raw.strip()
    if not raw:
        return list(default)
    return [float(x.strip()) for x in raw.split(",") if x.strip()]


def parse_int_list(raw: str, default: Sequence[int]) -> List[int]:
    raw = raw.strip()
    if not raw:
        return list(default)
    return [int(x.strip()) for x in raw.split(",") if x.strip()]


def atomic_write_text(path: Path, text: str, encoding: str = "utf-8") -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_name(path.name + ".tmp")
    tmp.write_text(text, encoding=encoding)
    os.replace(tmp, path)


def atomic_write_json(path: Path, obj: Any, indent: int = 2) -> None:
    atomic_write_text(path, json.dumps(obj, ensure_ascii=False, indent=indent))


def atomic_write_csv(path: Path, df: pd.DataFrame) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_name(path.name + ".tmp")
    df.to_csv(tmp, index=False)
    os.replace(tmp, path)


def atomic_torch_save(obj: Any, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_name(path.name + ".tmp")
    torch.save(obj, tmp)
    os.replace(tmp, path)


def find_project_root() -> Path:
    here = Path.cwd().resolve()
    for candidate in [here, *here.parents]:
        if candidate.name == "fruitnet-chili-yield":
            return candidate
    return here


PROJECT_ROOT = find_project_root()
os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# Keep the exact path convention used in 06_train.py.
DATA_DIR = Path(os.environ.get("DATA_DIR", "/mnt/f/fruitnet-chili-yield-data/finetune_data")).resolve()
OUTPUT_ROOT = Path(os.environ.get("OUTPUT_ROOT", "/mnt/f/fruitnet-chili-yield-outputs")).resolve()
DATASET_NAME = DATA_DIR.name

EXP_NAME = os.environ.get("EXP_NAME", f"exp03_rl_{DATASET_NAME}")
EXP_DIR = OUTPUT_ROOT / "results" / EXP_NAME
RUN_DIR = OUTPUT_ROOT / "runs" / "rl" / DATASET_NAME
CHECKPOINT_DIR = RUN_DIR / "checkpoints"
CACHE_DIR = EXP_DIR / "candidate_cache"
ARTIFACT_DIR = OUTPUT_ROOT / "artifacts" / "rl"
FINETUNED_ARTIFACT_DIR = OUTPUT_ROOT / "artifacts" / "finetuned"
LOG_DIR = OUTPUT_ROOT / "logs"
LOCAL_LOG_DIR = PROJECT_ROOT / "logs"
CONFIG_DATA_DIR = PROJECT_ROOT / "configs" / "data"

for d in [EXP_DIR, RUN_DIR, CHECKPOINT_DIR, CACHE_DIR, ARTIFACT_DIR, LOG_DIR, LOCAL_LOG_DIR, CONFIG_DATA_DIR]:
    d.mkdir(parents=True, exist_ok=True)

RUN_STAMP = datetime.now().strftime("%Y%m%d_%H%M%S")
LOG_PATH = LOG_DIR / f"07_rl_count_refine_{DATASET_NAME}_{RUN_STAMP}.log"
LOCAL_LOG_PATH = LOCAL_LOG_DIR / f"07_rl_count_refine_{DATASET_NAME}_{RUN_STAMP}.log"

logger = logging.getLogger("fruitnet_rl")
logger.setLevel(logging.INFO)
logger.handlers.clear()
_fmt = logging.Formatter("%(asctime)s | %(levelname)s | %(message)s")
for _path in [LOG_PATH, LOCAL_LOG_PATH]:
    fh = logging.FileHandler(_path, encoding="utf-8")
    fh.setFormatter(_fmt)
    logger.addHandler(fh)
sh = logging.StreamHandler(sys.stdout)
sh.setFormatter(_fmt)
logger.addHandler(sh)


# ============================================================
# 1. Config
# ============================================================

@dataclass
class RLConfig:
    # Runtime
    seed: int = env_int("SEED", 42)
    device: str = os.environ.get("DEVICE", "0")
    torch_device: str = os.environ.get("TORCH_DEVICE", "cuda" if torch.cuda.is_available() else "cpu")
    img_size: int = env_int("IMG_SIZE", 640)
    workers: int = env_int("WORKERS", 8)
    class_mode: str = os.environ.get("CLASS_MODE", "auto").strip()
    target_cls_env: str = os.environ.get("TARGET_CLS", "").strip()

    # Detector/candidate cache
    detector_key: str = os.environ.get("DETECTOR_KEY", "fruitnet_gcs").strip()
    detector_path_env: str = os.environ.get("DETECTOR_PATH", "").strip()
    rebuild_cache: bool = env_bool("REBUILD_CACHE", False)
    cache_splits: str = os.environ.get("RL_CACHE_SPLITS", "train,val,test").strip()
    cand_conf: float = env_float("CAND_CONF", 0.001)
    cand_iou: float = env_float("CAND_IOU", 0.99)
    max_det: int = env_int("MAX_DET", 1000)
    cache_batch: int = env_int("CACHE_BATCH", 1)

    # Baseline thresholds
    fixed_conf: float = env_float("FIXED_CONF", 0.25)
    fixed_iou: float = env_float("FIXED_IOU", 0.70)
    fixed_scale: float = env_float("FIXED_SCALE", 1.00)

    # Grid/action space
    conf_grid_raw: str = os.environ.get("CONF_GRID", "0.03,0.05,0.075,0.10,0.15,0.20,0.25,0.30,0.40,0.50")
    iou_grid_raw: str = os.environ.get("IOU_GRID", "0.30,0.45,0.55,0.65,0.70,0.80,0.90")
    scale_grid_raw: str = os.environ.get("SCALE_GRID", "0.85,0.90,0.95,1.00,1.05,1.10,1.15")
    soft_sigma_grid_raw: str = os.environ.get("SOFT_SIGMA_GRID", "0.25,0.50,0.75")

    # RL train/eval split design.
    # Default uses val for policy learning and test for final reporting, so detector training remains separated.
    rl_train_splits: str = os.environ.get("RL_TRAIN_SPLITS", "val").strip()
    policy_val_split: str = os.environ.get("POLICY_VAL_SPLIT", "val").strip()
    final_test_split: str = os.environ.get("FINAL_TEST_SPLIT", "test").strip()

    # PPO/contextual-bandit training
    run_train_rl: bool = env_bool("RUN_TRAIN_RL", True)
    run_final_eval: bool = env_bool("RUN_FINAL_EVAL", True)
    resume: bool = env_bool("RESUME", True)
    reset_rl: bool = env_bool("RESET_RL", False)
    rl_epochs: int = env_int("RL_EPOCHS", 300)
    batch_size: int = env_int("RL_BATCH_SIZE", 64)
    hidden_dim: int = env_int("RL_HIDDEN_DIM", 128)
    lr: float = env_float("RL_LR", 3e-4)
    weight_decay: float = env_float("RL_WEIGHT_DECAY", 1e-5)
    ppo_clip: float = env_float("PPO_CLIP", 0.20)
    ppo_updates: int = env_int("PPO_UPDATES", 4)
    entropy_coef: float = env_float("ENTROPY_COEF", 0.01)
    value_coef: float = env_float("VALUE_COEF", 0.50)
    grad_clip: float = env_float("GRAD_CLIP", 1.00)
    eval_every: int = env_int("EVAL_EVERY", 5)
    save_every: int = env_int("SAVE_EVERY", 1)

    # Reward weights. Primary goal is count error; penalties are secondary diagnostics.
    duplicate_penalty: float = env_float("DUPLICATE_PENALTY", 0.05)
    action_penalty: float = env_float("ACTION_PENALTY", 0.005)
    latency_penalty: float = env_float("LATENCY_PENALTY", 0.0)

    # Controls for quick debug runs
    max_cache_images: int = env_int("MAX_CACHE_IMAGES", 0)  # 0 means all
    max_train_records: int = env_int("MAX_TRAIN_RECORDS", 0)


CFG = RLConfig()
CONF_GRID = parse_float_list(CFG.conf_grid_raw, [0.05, 0.10, 0.15, 0.20, 0.25, 0.30, 0.40, 0.50])
IOU_GRID = parse_float_list(CFG.iou_grid_raw, [0.30, 0.45, 0.55, 0.65, 0.70, 0.80, 0.90])
SCALE_GRID = parse_float_list(CFG.scale_grid_raw, [0.90, 0.95, 1.0, 1.05, 1.10])
SOFT_SIGMA_GRID = parse_float_list(CFG.soft_sigma_grid_raw, [0.25, 0.50, 0.75])


def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


set_seed(CFG.seed)


# ============================================================
# 2. Dataset YAML and detector resolution
# ============================================================

def read_names_from_yaml(data_yaml: Path) -> List[str]:
    cfg = yaml.safe_load(data_yaml.read_text(encoding="utf-8")) or {}
    names = cfg.get("names", ["chili"])
    if isinstance(names, dict):
        return [names[k] for k in sorted(names, key=lambda x: int(x))]
    return list(names)


def resolve_data_yaml() -> Path:
    env_yaml = os.environ.get("DATA_YAML", "").strip()
    if env_yaml:
        p = Path(env_yaml).resolve()
        if not p.exists():
            raise FileNotFoundError(f"DATA_YAML was provided but does not exist: {p}")
        return p

    out = CONFIG_DATA_DIR / f"{DATASET_NAME}.yaml"
    if out.exists():
        return out

    src = DATA_DIR / "data.yaml"
    names: List[str] = ["chili"]
    if src.exists():
        try:
            raw = yaml.safe_load(src.read_text(encoding="utf-8")) or {}
            nm = raw.get("names")
            if isinstance(nm, dict):
                names = [nm[k] for k in sorted(nm, key=lambda x: int(x))]
            elif isinstance(nm, list) and nm:
                names = nm
        except Exception as e:
            logger.warning("Could not read %s; using default names: %s", src, repr(e))

    out.write_text(
        yaml.safe_dump(
            {
                "path": str(DATA_DIR),
                "train": "images/train",
                "val": "images/val",
                "test": "images/test",
                "nc": len(names),
                "names": {i: n for i, n in enumerate(names)},
            },
            sort_keys=False,
            allow_unicode=True,
        ),
        encoding="utf-8",
    )
    return out


DATA_YAML = resolve_data_yaml()
CLASS_NAMES = read_names_from_yaml(DATA_YAML)


def infer_target_cls(names: List[str]) -> int:
    if CFG.target_cls_env:
        return int(CFG.target_cls_env)
    if len(names) <= 1:
        return 0
    lowered = [str(n).lower() for n in names]
    for needle in ["red", "chili_red", "ripe", "chin", "chín", "do", "đỏ"]:
        for i, name in enumerate(lowered):
            if needle in name:
                return i
    return min(2, len(names) - 1)


TARGET_CLS = infer_target_cls(CLASS_NAMES)
TARGET_CLASS_NAME = CLASS_NAMES[TARGET_CLS] if TARGET_CLS < len(CLASS_NAMES) else str(TARGET_CLS)


def register_custom_modules() -> bool:
    try:
        from models.fruitnet.register_ultralytics_modules import register_fruitnet_modules
        register_fruitnet_modules()
        logger.info("Registered FruitNet custom modules for Ultralytics.")
        return True
    except Exception as e:
        logger.warning("Could not register FruitNet custom modules. Baseline YOLO may still work. Error=%s", repr(e))
        return False


def resolve_detector_path() -> Path:
    if CFG.detector_path_env:
        p = Path(CFG.detector_path_env).expanduser().resolve()
        if not p.exists():
            raise FileNotFoundError(f"DETECTOR_PATH does not exist: {p}")
        return p

    key = CFG.detector_key
    candidates = [
        FINETUNED_ARTIFACT_DIR / f"{key}_{DATASET_NAME}_best.pt",
        OUTPUT_ROOT / "runs" / "detectors" / DATASET_NAME / f"{key}_{DATASET_NAME}" / "weights" / "best.pt",
        OUTPUT_ROOT / "runs" / "detectors" / DATASET_NAME / f"{key}_{DATASET_NAME}" / "weights" / "last.pt",
        OUTPUT_ROOT / "runs" / "detectors" / "finetune_data" / f"{key}_finetune_data" / "weights" / "best.pt",
        OUTPUT_ROOT / "runs" / "detectors" / "finetune_data" / f"{key}_finetune_data" / "weights" / "last.pt",
    ]
    for p in candidates:
        if p.exists():
            return p.resolve()
    raise FileNotFoundError(
        "Could not resolve detector checkpoint. Set DETECTOR_PATH explicitly. Tried:\n" +
        "\n".join(str(p) for p in candidates)
    )


# ============================================================
# 3. Dataset records and candidate cache
# ============================================================

def data_cfg() -> Dict[str, Any]:
    raw = yaml.safe_load(DATA_YAML.read_text(encoding="utf-8")) or {}
    root = Path(raw.get("path", DATA_DIR)).resolve()
    raw["path"] = str(root)
    return raw


def iter_images(path: Path) -> Iterable[Path]:
    if not path.exists():
        return []
    return sorted(p for p in path.rglob("*") if p.is_file() and p.suffix.lower() in IMG_EXTS)


def split_image_paths(split: str) -> List[Path]:
    cfg = data_cfg()
    root = Path(cfg["path"])
    rel = cfg.get(split, f"images/{split}")
    return list(iter_images(root / rel))


def split_label_path(img_path: Path, split: str) -> Path:
    cfg = data_cfg()
    root = Path(cfg["path"])
    # Standard YOLO layout: images/<split>/name.jpg -> labels/<split>/name.txt
    try:
        rel = img_path.relative_to(root)
        parts = list(rel.parts)
        if "images" in parts:
            idx = parts.index("images")
            parts[idx] = "labels"
            return root / Path(*parts).with_suffix(".txt")
    except Exception:
        pass
    return root / "labels" / split / f"{img_path.stem}.txt"


def read_image_size(img_path: Path) -> Tuple[int, int]:
    with Image.open(img_path) as im:
        return im.width, im.height


def read_yolo_boxes(label_path: Path, target_cls: int, width: int, height: int) -> List[List[float]]:
    boxes: List[List[float]] = []
    if not label_path.exists():
        return boxes
    for line in label_path.read_text(encoding="utf-8", errors="ignore").splitlines():
        parts = line.strip().split()
        if len(parts) < 5:
            continue
        try:
            cls = int(float(parts[0]))
            if cls != target_cls:
                continue
            cx, cy, bw, bh = [float(x) for x in parts[1:5]]
            x1 = max(0.0, (cx - bw / 2.0) * width)
            y1 = max(0.0, (cy - bh / 2.0) * height)
            x2 = min(float(width), (cx + bw / 2.0) * width)
            y2 = min(float(height), (cy + bh / 2.0) * height)
            boxes.append([x1, y1, x2, y2])
        except Exception:
            continue
    return boxes


def build_split_records(split: str) -> List[Dict[str, Any]]:
    rows: List[Dict[str, Any]] = []
    imgs = split_image_paths(split)
    if CFG.max_cache_images and len(imgs) > CFG.max_cache_images:
        imgs = imgs[: CFG.max_cache_images]
    for img in imgs:
        try:
            w, h = read_image_size(img)
        except Exception as e:
            logger.warning("Skip unreadable image %s: %s", img, repr(e))
            continue
        lbl = split_label_path(img, split)
        gt_boxes = read_yolo_boxes(lbl, TARGET_CLS, w, h)
        rows.append(
            {
                "split": split,
                "image": str(img),
                "image_name": img.name,
                "label": str(lbl),
                "width": w,
                "height": h,
                "gt_count": len(gt_boxes),
                "gt_boxes": gt_boxes,
            }
        )
    return rows


def cache_paths(split: str) -> Tuple[Path, Path]:
    return CACHE_DIR / f"{split}_candidates.jsonl", CACHE_DIR / f"{split}_candidates_meta.json"


def load_jsonl(path: Path) -> List[Dict[str, Any]]:
    rows: List[Dict[str, Any]] = []
    with path.open("r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows


def write_jsonl_atomic(path: Path, rows: Sequence[Dict[str, Any]]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_name(path.name + ".tmp")
    with tmp.open("w", encoding="utf-8") as f:
        for row in rows:
            f.write(json.dumps(row, ensure_ascii=False) + "\n")
    os.replace(tmp, path)


def cache_is_valid(meta_path: Path, detector_path: Path, split: str) -> bool:
    if CFG.rebuild_cache:
        return False
    jsonl_path, _ = cache_paths(split)
    if not jsonl_path.exists() or not meta_path.exists():
        return False
    try:
        meta = json.loads(meta_path.read_text(encoding="utf-8"))
        return (
            meta.get("detector_path") == str(detector_path)
            and meta.get("target_cls") == TARGET_CLS
            and meta.get("cand_conf") == CFG.cand_conf
            and meta.get("cand_iou") == CFG.cand_iou
            and meta.get("img_size") == CFG.img_size
            and meta.get("max_det") == CFG.max_det
        )
    except Exception:
        return False


def build_or_load_candidate_cache(split: str, detector_path: Path) -> List[Dict[str, Any]]:
    jsonl_path, meta_path = cache_paths(split)
    if cache_is_valid(meta_path, detector_path, split):
        logger.info("Using existing candidate cache for split=%s: %s", split, jsonl_path)
        return load_jsonl(jsonl_path)

    records = build_split_records(split)
    if not records:
        logger.warning("No images found for split=%s; cache skipped.", split)
        return []

    from ultralytics import YOLO

    logger.info(
        "Building candidate cache split=%s images=%d detector=%s conf=%.4f iou=%.2f max_det=%d",
        split, len(records), detector_path, CFG.cand_conf, CFG.cand_iou, CFG.max_det,
    )
    model = YOLO(str(detector_path))
    out_rows: List[Dict[str, Any]] = []
    started = time.time()

    for rec in tqdm(records, desc=f"cache:{split}"):
        t0 = time.time()
        try:
            pred = model.predict(
                str(rec["image"]),
                imgsz=CFG.img_size,
                conf=CFG.cand_conf,
                iou=CFG.cand_iou,
                max_det=CFG.max_det,
                device=CFG.device,
                verbose=False,
            )[0]
            boxes = []
            confs = []
            classes = []
            if pred.boxes is not None and pred.boxes.xyxy is not None:
                boxes = pred.boxes.xyxy.detach().cpu().numpy().astype(float).tolist()
                confs = pred.boxes.conf.detach().cpu().numpy().astype(float).tolist()
                classes = pred.boxes.cls.detach().cpu().numpy().astype(int).tolist()
            rec.update(
                {
                    "pred_boxes": boxes,
                    "pred_confs": confs,
                    "pred_classes": classes,
                    "candidate_count": len(boxes),
                    "candidate_target_count": int(sum(1 for c in classes if int(c) == TARGET_CLS)),
                    "predict_ms": (time.time() - t0) * 1000.0,
                    "cache_error": "",
                }
            )
        except Exception as e:
            rec.update(
                {
                    "pred_boxes": [],
                    "pred_confs": [],
                    "pred_classes": [],
                    "candidate_count": 0,
                    "candidate_target_count": 0,
                    "predict_ms": (time.time() - t0) * 1000.0,
                    "cache_error": repr(e),
                }
            )
            logger.exception("Prediction failed for image=%s", rec.get("image"))
        out_rows.append(rec)

        # Frequent partial checkpoint for cache construction.
        if len(out_rows) % 50 == 0:
            write_jsonl_atomic(jsonl_path, out_rows)
            atomic_write_json(
                meta_path,
                {
                    "status": "partial",
                    "split": split,
                    "rows_written": len(out_rows),
                    "detector_path": str(detector_path),
                    "target_cls": TARGET_CLS,
                    "target_class_name": TARGET_CLASS_NAME,
                    "cand_conf": CFG.cand_conf,
                    "cand_iou": CFG.cand_iou,
                    "img_size": CFG.img_size,
                    "max_det": CFG.max_det,
                    "updated_at": datetime.now().isoformat(timespec="seconds"),
                },
            )

    write_jsonl_atomic(jsonl_path, out_rows)
    elapsed = time.time() - started
    atomic_write_json(
        meta_path,
        {
            "status": "complete",
            "split": split,
            "n_images": len(out_rows),
            "detector_path": str(detector_path),
            "target_cls": TARGET_CLS,
            "target_class_name": TARGET_CLASS_NAME,
            "cand_conf": CFG.cand_conf,
            "cand_iou": CFG.cand_iou,
            "img_size": CFG.img_size,
            "max_det": CFG.max_det,
            "elapsed_sec": elapsed,
            "avg_predict_ms": float(np.mean([r.get("predict_ms", 0.0) for r in out_rows])) if out_rows else None,
            "created_at": datetime.now().isoformat(timespec="seconds"),
        },
    )
    logger.info("Candidate cache complete split=%s -> %s", split, jsonl_path)
    return out_rows


# ============================================================
# 4. NMS, Soft-NMS, counting, and metrics
# ============================================================

def _np_boxes(record: Dict[str, Any], target_only: bool = True) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
    boxes = np.asarray(record.get("pred_boxes", []), dtype=np.float32).reshape(-1, 4)
    confs = np.asarray(record.get("pred_confs", []), dtype=np.float32).reshape(-1)
    classes = np.asarray(record.get("pred_classes", []), dtype=np.int64).reshape(-1)
    if len(boxes) == 0:
        return boxes, confs, classes
    if target_only:
        mask = classes == TARGET_CLS
        return boxes[mask], confs[mask], classes[mask]
    return boxes, confs, classes


def box_iou_one_to_many(box: np.ndarray, boxes: np.ndarray) -> np.ndarray:
    if boxes.size == 0:
        return np.zeros((0,), dtype=np.float32)
    x1 = np.maximum(box[0], boxes[:, 0])
    y1 = np.maximum(box[1], boxes[:, 1])
    x2 = np.minimum(box[2], boxes[:, 2])
    y2 = np.minimum(box[3], boxes[:, 3])
    inter = np.maximum(0.0, x2 - x1) * np.maximum(0.0, y2 - y1)
    area1 = max(0.0, float(box[2] - box[0])) * max(0.0, float(box[3] - box[1]))
    area2 = np.maximum(0.0, boxes[:, 2] - boxes[:, 0]) * np.maximum(0.0, boxes[:, 3] - boxes[:, 1])
    union = area1 + area2 - inter + 1e-9
    return inter / union


def pairwise_iou(boxes: np.ndarray) -> np.ndarray:
    n = len(boxes)
    out = np.zeros((n, n), dtype=np.float32)
    for i in range(n):
        out[i] = box_iou_one_to_many(boxes[i], boxes)
    return out


def hard_nms_indices(boxes: np.ndarray, scores: np.ndarray, iou_thr: float) -> List[int]:
    if len(boxes) == 0:
        return []
    order = scores.argsort()[::-1]
    keep: List[int] = []
    while order.size > 0:
        i = int(order[0])
        keep.append(i)
        if order.size == 1:
            break
        ious = box_iou_one_to_many(boxes[i], boxes[order[1:]])
        order = order[1:][ious <= iou_thr]
    return keep


def soft_nms_indices(
    boxes: np.ndarray,
    scores: np.ndarray,
    iou_thr: float = 0.50,
    sigma: float = 0.50,
    score_thr: float = 0.25,
    method: str = "gaussian",
) -> List[int]:
    if len(boxes) == 0:
        return []
    boxes = boxes.copy()
    scores = scores.copy()
    idxs = np.arange(len(scores))
    keep: List[int] = []
    while len(scores) > 0:
        m = int(np.argmax(scores))
        best_score = float(scores[m])
        if best_score < score_thr:
            break
        keep.append(int(idxs[m]))

        best_box = boxes[m].copy()
        boxes = np.delete(boxes, m, axis=0)
        scores = np.delete(scores, m, axis=0)
        idxs = np.delete(idxs, m, axis=0)
        if len(scores) == 0:
            break
        ious = box_iou_one_to_many(best_box, boxes)
        if method == "linear":
            decay = np.ones_like(scores)
            decay[ious > iou_thr] = 1.0 - ious[ious > iou_thr]
        else:
            decay = np.exp(-(ious * ious) / max(sigma, 1e-6))
        scores *= decay
    return keep


def duplicate_iou_ratio(boxes: np.ndarray, iou_thr: float = 0.75) -> float:
    if len(boxes) < 2:
        return 0.0
    mat = pairwise_iou(boxes)
    tri = mat[np.triu_indices(len(boxes), k=1)]
    return float((tri > iou_thr).sum() / max(1, len(tri)))


def count_with_threshold(record: Dict[str, Any], conf: float, iou: float, scale: float = 1.0) -> Dict[str, Any]:
    boxes, scores, _ = _np_boxes(record, target_only=True)
    if len(boxes) == 0:
        return {"count_raw": 0, "pred_count": 0, "kept_boxes": [], "duplicate_iou_ratio": 0.0, "latency_ms": 0.0}
    t0 = time.time()
    mask = scores >= conf
    boxes_f = boxes[mask]
    scores_f = scores[mask]
    keep = hard_nms_indices(boxes_f, scores_f, iou)
    kept = boxes_f[keep] if keep else np.zeros((0, 4), dtype=np.float32)
    raw = len(kept)
    pred = int(max(0, round(raw * scale)))
    return {
        "count_raw": int(raw),
        "pred_count": pred,
        "kept_boxes": kept.tolist(),
        "duplicate_iou_ratio": duplicate_iou_ratio(kept),
        "latency_ms": (time.time() - t0) * 1000.0,
    }


def count_with_soft_nms(record: Dict[str, Any], conf: float, iou: float, sigma: float, scale: float = 1.0) -> Dict[str, Any]:
    boxes, scores, _ = _np_boxes(record, target_only=True)
    if len(boxes) == 0:
        return {"count_raw": 0, "pred_count": 0, "kept_boxes": [], "duplicate_iou_ratio": 0.0, "latency_ms": 0.0}
    t0 = time.time()
    keep = soft_nms_indices(boxes, scores, iou_thr=iou, sigma=sigma, score_thr=conf, method="gaussian")
    kept = boxes[keep] if keep else np.zeros((0, 4), dtype=np.float32)
    raw = len(kept)
    pred = int(max(0, round(raw * scale)))
    return {
        "count_raw": int(raw),
        "pred_count": pred,
        "kept_boxes": kept.tolist(),
        "duplicate_iou_ratio": duplicate_iou_ratio(kept),
        "latency_ms": (time.time() - t0) * 1000.0,
    }


def count_metrics(gt: Sequence[int], pred: Sequence[int], duplicate_ratio: Optional[Sequence[float]] = None, latency_ms: Optional[Sequence[float]] = None) -> Dict[str, float]:
    g = np.asarray(gt, dtype=np.float32)
    p = np.asarray(pred, dtype=np.float32)
    if len(g) == 0:
        return {
            "n_images": 0,
            "MAE_count": math.nan,
            "RMSE_count": math.nan,
            "MAPE_count": math.nan,
            "Counting_Accuracy": math.nan,
            "bias": math.nan,
            "over_count_error": math.nan,
            "missed_count_error": math.nan,
            "exact_match_rate": math.nan,
            "median_abs_error": math.nan,
            "p90_abs_error": math.nan,
            "duplicate_iou_ratio": math.nan,
            "postprocess_latency_ms": math.nan,
        }
    err = p - g
    abs_err = np.abs(err)
    mape = float(np.mean(abs_err / np.maximum(g, 1.0)) * 100.0)
    return {
        "n_images": int(len(g)),
        "MAE_count": float(abs_err.mean()),
        "RMSE_count": float(np.sqrt((err ** 2).mean())),
        "MAPE_count": mape,
        "Counting_Accuracy": max(0.0, 100.0 - mape),
        "bias": float(err.mean()),
        "over_count_error": float(np.maximum(err, 0).mean()),
        "missed_count_error": float(np.maximum(-err, 0).mean()),
        "exact_match_rate": float((abs_err == 0).mean()),
        "median_abs_error": float(np.median(abs_err)),
        "p90_abs_error": float(np.percentile(abs_err, 90)),
        "duplicate_iou_ratio": float(np.mean(duplicate_ratio)) if duplicate_ratio is not None and len(duplicate_ratio) else math.nan,
        "postprocess_latency_ms": float(np.mean(latency_ms)) if latency_ms is not None and len(latency_ms) else math.nan,
    }


def evaluate_fixed(records: Sequence[Dict[str, Any]], conf: float, iou: float, scale: float = 1.0) -> Tuple[Dict[str, float], pd.DataFrame]:
    rows = []
    for r in records:
        out = count_with_threshold(r, conf, iou, scale)
        gt = int(r.get("gt_count", 0))
        pred = int(out["pred_count"])
        rows.append(
            {
                "image": r.get("image"),
                "image_name": r.get("image_name"),
                "split": r.get("split"),
                "gt_count": gt,
                "pred_count": pred,
                "count_raw": out["count_raw"],
                "abs_err": abs(pred - gt),
                "sq_err": (pred - gt) ** 2,
                "ape": abs(pred - gt) / max(gt, 1),
                "conf": conf,
                "iou": iou,
                "scale": scale,
                "duplicate_iou_ratio": out["duplicate_iou_ratio"],
                "postprocess_latency_ms": out["latency_ms"],
            }
        )
    df = pd.DataFrame(rows)
    metrics = count_metrics(df["gt_count"].tolist(), df["pred_count"].tolist(), df["duplicate_iou_ratio"].tolist(), df["postprocess_latency_ms"].tolist()) if len(df) else count_metrics([], [])
    return metrics, df


def evaluate_soft(records: Sequence[Dict[str, Any]], conf: float, iou: float, sigma: float, scale: float = 1.0) -> Tuple[Dict[str, float], pd.DataFrame]:
    rows = []
    for r in records:
        out = count_with_soft_nms(r, conf, iou, sigma, scale)
        gt = int(r.get("gt_count", 0))
        pred = int(out["pred_count"])
        rows.append(
            {
                "image": r.get("image"),
                "image_name": r.get("image_name"),
                "split": r.get("split"),
                "gt_count": gt,
                "pred_count": pred,
                "count_raw": out["count_raw"],
                "abs_err": abs(pred - gt),
                "sq_err": (pred - gt) ** 2,
                "ape": abs(pred - gt) / max(gt, 1),
                "conf": conf,
                "iou": iou,
                "sigma": sigma,
                "scale": scale,
                "duplicate_iou_ratio": out["duplicate_iou_ratio"],
                "postprocess_latency_ms": out["latency_ms"],
            }
        )
    df = pd.DataFrame(rows)
    metrics = count_metrics(df["gt_count"].tolist(), df["pred_count"].tolist(), df["duplicate_iou_ratio"].tolist(), df["postprocess_latency_ms"].tolist()) if len(df) else count_metrics([], [])
    return metrics, df


def tune_grid(records: Sequence[Dict[str, Any]], method: str = "hard") -> Tuple[Dict[str, Any], pd.DataFrame]:
    rows = []
    if method == "hard":
        for conf in CONF_GRID:
            for iou in IOU_GRID:
                for scale in SCALE_GRID:
                    metrics, _ = evaluate_fixed(records, conf, iou, scale)
                    row = {"method": "validation_grid_search", "conf": conf, "iou": iou, "scale": scale, **metrics}
                    rows.append(row)
    elif method == "soft":
        for conf in CONF_GRID:
            for iou in IOU_GRID:
                for sigma in SOFT_SIGMA_GRID:
                    for scale in SCALE_GRID:
                        metrics, _ = evaluate_soft(records, conf, iou, sigma, scale)
                        row = {"method": "soft_nms_tuned", "conf": conf, "iou": iou, "sigma": sigma, "scale": scale, **metrics}
                        rows.append(row)
    else:
        raise ValueError(method)
    df = pd.DataFrame(rows)
    if df.empty:
        return {}, df
    df = df.sort_values(["MAE_count", "RMSE_count", "MAPE_count"], ascending=True).reset_index(drop=True)
    return df.iloc[0].to_dict(), df


# ============================================================
# 5. State encoder and action space
# ============================================================

FEATURE_NAMES = [
    "n_all_norm",
    "n_target_norm",
    "target_ratio",
    "conf_mean",
    "conf_std",
    "conf_min",
    "conf_max",
    "conf_q25",
    "conf_q50",
    "conf_q75",
    "area_mean",
    "area_std",
    "area_min",
    "area_max",
    "small_box_ratio",
    "density_per_mpix",
    "target_density_per_mpix",
    "overlap_ratio_iou50",
    "overlap_ratio_iou75",
    "mean_pair_iou",
    "max_pair_iou",
    "image_aspect",
    "gt_free_candidate_count_norm",
]


def safe_stats(values: np.ndarray) -> Tuple[float, float, float, float, float, float, float]:
    if values.size == 0:
        return 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0
    return (
        float(values.mean()),
        float(values.std()),
        float(values.min()),
        float(values.max()),
        float(np.quantile(values, 0.25)),
        float(np.quantile(values, 0.50)),
        float(np.quantile(values, 0.75)),
    )


def encode_state_raw(record: Dict[str, Any]) -> np.ndarray:
    boxes_all, conf_all, classes_all = _np_boxes(record, target_only=False)
    boxes_t, conf_t, _ = _np_boxes(record, target_only=True)
    w = max(float(record.get("width", 1)), 1.0)
    h = max(float(record.get("height", 1)), 1.0)
    img_area = max(w * h, 1.0)
    n_all = len(boxes_all)
    n_t = len(boxes_t)
    target_ratio = float(n_t / max(n_all, 1))

    c_mean, c_std, c_min, c_max, c_q25, c_q50, c_q75 = safe_stats(conf_t)
    if n_t:
        areas = np.maximum(0.0, boxes_t[:, 2] - boxes_t[:, 0]) * np.maximum(0.0, boxes_t[:, 3] - boxes_t[:, 1]) / img_area
        a_mean, a_std, a_min, a_max, _, _, _ = safe_stats(areas)
        small_ratio = float((areas < 0.0025).mean())  # proxy for small/far/partly visible fruit
        mat = pairwise_iou(boxes_t) if n_t >= 2 else np.zeros((n_t, n_t), dtype=np.float32)
        if n_t >= 2:
            tri = mat[np.triu_indices(n_t, k=1)]
            overlap50 = float((tri > 0.50).mean())
            overlap75 = float((tri > 0.75).mean())
            mean_iou = float(tri.mean())
            max_iou = float(tri.max())
        else:
            overlap50 = overlap75 = mean_iou = max_iou = 0.0
    else:
        a_mean = a_std = a_min = a_max = small_ratio = 0.0
        overlap50 = overlap75 = mean_iou = max_iou = 0.0

    density = float(n_all / (img_area / 1_000_000.0))
    target_density = float(n_t / (img_area / 1_000_000.0))
    aspect = float(w / h)
    no_gt_norm = float(max(0, n_t - int(record.get("gt_count", 0))) / 200.0)

    feats = np.asarray(
        [
            min(n_all / 200.0, 5.0),
            min(n_t / 200.0, 5.0),
            target_ratio,
            c_mean,
            c_std,
            c_min,
            c_max,
            c_q25,
            c_q50,
            c_q75,
            a_mean,
            a_std,
            a_min,
            a_max,
            small_ratio,
            min(density / 500.0, 10.0),
            min(target_density / 500.0, 10.0),
            overlap50,
            overlap75,
            mean_iou,
            max_iou,
            aspect,
            no_gt_norm,
        ],
        dtype=np.float32,
    )
    return feats


def compute_state_matrix(records: Sequence[Dict[str, Any]]) -> np.ndarray:
    if not records:
        return np.zeros((0, len(FEATURE_NAMES)), dtype=np.float32)
    return np.stack([encode_state_raw(r) for r in records]).astype(np.float32)


def fit_state_norm(states: np.ndarray) -> Dict[str, Any]:
    mean = states.mean(axis=0) if len(states) else np.zeros((states.shape[1],), dtype=np.float32)
    std = states.std(axis=0) if len(states) else np.ones((states.shape[1],), dtype=np.float32)
    std = np.where(std < 1e-6, 1.0, std)
    return {"mean": mean.astype(float).tolist(), "std": std.astype(float).tolist(), "feature_names": FEATURE_NAMES}


def apply_state_norm(states: np.ndarray, norm: Dict[str, Any]) -> np.ndarray:
    mean = np.asarray(norm["mean"], dtype=np.float32)
    std = np.asarray(norm["std"], dtype=np.float32)
    return ((states - mean) / std).astype(np.float32)


def build_action_space() -> List[Dict[str, float]]:
    actions = []
    for conf in CONF_GRID:
        for iou in IOU_GRID:
            for scale in SCALE_GRID:
                actions.append({"conf": float(conf), "iou": float(iou), "scale": float(scale)})
    return actions


ACTION_SPACE = build_action_space()
ACTION_SPACE_PATH = ARTIFACT_DIR / f"rl_action_space_{DATASET_NAME}.json"
atomic_write_json(ACTION_SPACE_PATH, ACTION_SPACE)


# ============================================================
# 6. Actor-Critic policy and PPO training
# ============================================================

class ActorCritic(nn.Module):
    def __init__(self, state_dim: int, n_actions: int, hidden_dim: int = 128):
        super().__init__()
        self.backbone = nn.Sequential(
            nn.Linear(state_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.ReLU(inplace=True),
            nn.Linear(hidden_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.ReLU(inplace=True),
        )
        self.policy = nn.Linear(hidden_dim, n_actions)
        self.value = nn.Linear(hidden_dim, 1)

    def forward(self, x: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
        h = self.backbone(x)
        return self.policy(h), self.value(h).squeeze(-1)


def action_to_count(record: Dict[str, Any], action_idx: int) -> Dict[str, Any]:
    a = ACTION_SPACE[int(action_idx)]
    out = count_with_threshold(record, a["conf"], a["iou"], a["scale"])
    out.update({"action_idx": int(action_idx), "conf": a["conf"], "iou": a["iou"], "scale": a["scale"]})
    return out


def reward_for(record: Dict[str, Any], action_idx: int) -> float:
    out = action_to_count(record, action_idx)
    gt = int(record.get("gt_count", 0))
    pred = int(out["pred_count"])
    rel_abs = abs(pred - gt) / max(gt, 1)
    dup_pen = CFG.duplicate_penalty * float(out.get("duplicate_iou_ratio", 0.0))
    a = ACTION_SPACE[int(action_idx)]
    # Penalize aggressive count scaling slightly so the policy first learns threshold/NMS behavior.
    scale_pen = CFG.action_penalty * abs(float(a["scale"]) - 1.0)
    lat_pen = CFG.latency_penalty * float(out.get("latency_ms", 0.0)) / 100.0
    return float(-rel_abs - dup_pen - scale_pen - lat_pen)


def evaluate_policy(
    policy: ActorCritic,
    records: Sequence[Dict[str, Any]],
    state_norm: Dict[str, Any],
    split_name: str,
    deterministic: bool = True,
) -> Tuple[Dict[str, float], pd.DataFrame]:
    if not records:
        return count_metrics([], []), pd.DataFrame()
    policy.eval()
    states = apply_state_norm(compute_state_matrix(records), state_norm)
    device = next(policy.parameters()).device
    rows = []
    with torch.no_grad():
        for i, rec in enumerate(records):
            s = torch.tensor(states[i : i + 1], dtype=torch.float32, device=device)
            logits, value = policy(s)
            if deterministic:
                action_idx = int(torch.argmax(logits, dim=-1).item())
            else:
                action_idx = int(torch.distributions.Categorical(logits=logits).sample().item())
            out = action_to_count(rec, action_idx)
            gt = int(rec.get("gt_count", 0))
            pred = int(out["pred_count"])
            rows.append(
                {
                    "split": split_name,
                    "image": rec.get("image"),
                    "image_name": rec.get("image_name"),
                    "gt_count": gt,
                    "pred_count": pred,
                    "count_raw": out["count_raw"],
                    "abs_err": abs(pred - gt),
                    "sq_err": (pred - gt) ** 2,
                    "ape": abs(pred - gt) / max(gt, 1),
                    "action_idx": action_idx,
                    "conf": out["conf"],
                    "iou": out["iou"],
                    "scale": out["scale"],
                    "value": float(value.item()),
                    "duplicate_iou_ratio": float(out.get("duplicate_iou_ratio", 0.0)),
                    "postprocess_latency_ms": float(out.get("latency_ms", 0.0)),
                    "candidate_target_count": int(rec.get("candidate_target_count", 0)),
                    "gt_minus_pred": gt - pred,
                    "pred_minus_gt": pred - gt,
                }
            )
    df = pd.DataFrame(rows)
    metrics = count_metrics(df["gt_count"].tolist(), df["pred_count"].tolist(), df["duplicate_iou_ratio"].tolist(), df["postprocess_latency_ms"].tolist())
    return metrics, df


def checkpoint_payload(
    policy: ActorCritic,
    optimizer: torch.optim.Optimizer,
    epoch: int,
    best_val_mae: float,
    state_norm: Dict[str, Any],
    history: List[Dict[str, Any]],
    detector_path: Path,
) -> Dict[str, Any]:
    return {
        "epoch": int(epoch),
        "best_val_mae": float(best_val_mae),
        "model_state": policy.state_dict(),
        "optimizer_state": optimizer.state_dict(),
        "state_norm": state_norm,
        "action_space": ACTION_SPACE,
        "feature_names": FEATURE_NAMES,
        "config": asdict(CFG),
        "target_cls": TARGET_CLS,
        "target_class_name": TARGET_CLASS_NAME,
        "class_names": CLASS_NAMES,
        "detector_path": str(detector_path),
        "history": history,
        "saved_at": datetime.now().isoformat(timespec="seconds"),
    }


def load_checkpoint_if_available(policy: ActorCritic, optimizer: torch.optim.Optimizer) -> Tuple[int, float, Optional[Dict[str, Any]], List[Dict[str, Any]]]:
    if CFG.reset_rl or not CFG.resume:
        return 0, float("inf"), None, []
    latest = CHECKPOINT_DIR / "policy_latest.pt"
    if not latest.exists():
        return 0, float("inf"), None, []
    logger.info("Resuming RL checkpoint: %s", latest)
    ckpt = torch.load(latest, map_location=CFG.torch_device)
    policy.load_state_dict(ckpt["model_state"])
    optimizer.load_state_dict(ckpt["optimizer_state"])
    return int(ckpt.get("epoch", 0)), float(ckpt.get("best_val_mae", float("inf"))), ckpt.get("state_norm"), list(ckpt.get("history", []))


def train_rl_policy(
    train_records: Sequence[Dict[str, Any]],
    val_records: Sequence[Dict[str, Any]],
    detector_path: Path,
) -> Tuple[ActorCritic, Dict[str, Any], List[Dict[str, Any]]]:
    if not train_records:
        raise RuntimeError("No RL training records. Check RL_TRAIN_SPLITS and cache generation.")

    if CFG.max_train_records and len(train_records) > CFG.max_train_records:
        train_records = list(train_records)[: CFG.max_train_records]

    raw_states = compute_state_matrix(train_records)
    state_norm = fit_state_norm(raw_states)
    train_states = apply_state_norm(raw_states, state_norm)

    device = torch.device(CFG.torch_device)
    policy = ActorCritic(state_dim=train_states.shape[1], n_actions=len(ACTION_SPACE), hidden_dim=CFG.hidden_dim).to(device)
    optimizer = torch.optim.AdamW(policy.parameters(), lr=CFG.lr, weight_decay=CFG.weight_decay)

    start_epoch, best_val_mae, loaded_norm, history = load_checkpoint_if_available(policy, optimizer)
    if loaded_norm is not None:
        state_norm = loaded_norm
        train_states = apply_state_norm(raw_states, state_norm)

    logger.info(
        "RL train start: records=%d val_records=%d start_epoch=%d epochs=%d actions=%d state_dim=%d device=%s",
        len(train_records), len(val_records), start_epoch, CFG.rl_epochs, len(ACTION_SPACE), train_states.shape[1], device,
    )

    idx_all = np.arange(len(train_records))
    latest_ckpt = CHECKPOINT_DIR / "policy_latest.pt"
    best_ckpt = CHECKPOINT_DIR / "policy_best.pt"
    history_csv = EXP_DIR / "rl_training_history.csv"

    for epoch in range(start_epoch + 1, CFG.rl_epochs + 1):
        t_epoch = time.time()
        policy.train()
        np.random.shuffle(idx_all)
        epoch_losses: List[float] = []
        epoch_rewards: List[float] = []
        epoch_entropies: List[float] = []

        for start in range(0, len(idx_all), CFG.batch_size):
            batch_idx = idx_all[start : start + CFG.batch_size]
            s_np = train_states[batch_idx]
            s = torch.tensor(s_np, dtype=torch.float32, device=device)

            with torch.no_grad():
                logits_old, values_old = policy(s)
                dist_old = torch.distributions.Categorical(logits=logits_old)
                actions = dist_old.sample()
                old_logp = dist_old.log_prob(actions)
                rewards_np = np.asarray([reward_for(train_records[int(i)], int(a)) for i, a in zip(batch_idx, actions.detach().cpu().numpy())], dtype=np.float32)
                returns = torch.tensor(rewards_np, dtype=torch.float32, device=device)
                adv = returns - values_old
                if adv.numel() > 1:
                    adv = (adv - adv.mean()) / (adv.std() + 1e-8)

            for _ in range(CFG.ppo_updates):
                logits, values = policy(s)
                dist = torch.distributions.Categorical(logits=logits)
                logp = dist.log_prob(actions)
                ratio = torch.exp(logp - old_logp)
                unclipped = ratio * adv
                clipped = torch.clamp(ratio, 1.0 - CFG.ppo_clip, 1.0 + CFG.ppo_clip) * adv
                policy_loss = -torch.min(unclipped, clipped).mean()
                value_loss = F.mse_loss(values, returns)
                entropy = dist.entropy().mean()
                loss = policy_loss + CFG.value_coef * value_loss - CFG.entropy_coef * entropy

                optimizer.zero_grad(set_to_none=True)
                loss.backward()
                if CFG.grad_clip > 0:
                    nn.utils.clip_grad_norm_(policy.parameters(), CFG.grad_clip)
                optimizer.step()

                epoch_losses.append(float(loss.detach().cpu().item()))
                epoch_entropies.append(float(entropy.detach().cpu().item()))
            epoch_rewards.extend(rewards_np.astype(float).tolist())

        # Evaluate periodically and always at last epoch.
        do_eval = epoch == 1 or epoch % CFG.eval_every == 0 or epoch == CFG.rl_epochs
        val_metrics = {}
        if do_eval and val_records:
            val_metrics, val_df = evaluate_policy(policy, val_records, state_norm, split_name=CFG.policy_val_split, deterministic=True)
            atomic_write_csv(EXP_DIR / "rl_val_predictions_latest.csv", val_df)
            current_val_mae = float(val_metrics.get("MAE_count", float("inf")))
        else:
            current_val_mae = float("inf")

        row = {
            "epoch": epoch,
            "train_reward_mean": float(np.mean(epoch_rewards)) if epoch_rewards else math.nan,
            "train_reward_std": float(np.std(epoch_rewards)) if epoch_rewards else math.nan,
            "loss_mean": float(np.mean(epoch_losses)) if epoch_losses else math.nan,
            "entropy_mean": float(np.mean(epoch_entropies)) if epoch_entropies else math.nan,
            "elapsed_sec": time.time() - t_epoch,
            **{f"val_{k}": v for k, v in val_metrics.items()},
        }
        history.append(row)
        atomic_write_csv(history_csv, pd.DataFrame(history))

        if current_val_mae < best_val_mae:
            best_val_mae = current_val_mae
            atomic_torch_save(checkpoint_payload(policy, optimizer, epoch, best_val_mae, state_norm, history, detector_path), best_ckpt)
            shutil.copy2(best_ckpt, ARTIFACT_DIR / f"rl_policy_{DATASET_NAME}_best.pt")
            logger.info("New best RL policy at epoch=%d val_MAE=%.4f -> %s", epoch, best_val_mae, best_ckpt)

        if epoch % CFG.save_every == 0 or epoch == CFG.rl_epochs:
            atomic_torch_save(checkpoint_payload(policy, optimizer, epoch, best_val_mae, state_norm, history, detector_path), latest_ckpt)
            atomic_write_json(
                CHECKPOINT_DIR / "checkpoint_manifest.json",
                {
                    "latest": str(latest_ckpt),
                    "best": str(best_ckpt) if best_ckpt.exists() else "",
                    "latest_epoch": epoch,
                    "best_val_mae": best_val_mae,
                    "updated_at": datetime.now().isoformat(timespec="seconds"),
                    "log_path": str(LOG_PATH),
                },
            )

        logger.info(
            "RL epoch %03d/%03d | reward=%.4f | loss=%.4f | val_MAE=%s | best_MAE=%.4f | %.1fs",
            epoch,
            CFG.rl_epochs,
            float(np.mean(epoch_rewards)) if epoch_rewards else math.nan,
            float(np.mean(epoch_losses)) if epoch_losses else math.nan,
            f"{current_val_mae:.4f}" if do_eval and val_records else "NA",
            best_val_mae,
            time.time() - t_epoch,
        )

    # Load best policy for downstream evaluation when possible.
    best_ckpt = CHECKPOINT_DIR / "policy_best.pt"
    if best_ckpt.exists():
        ckpt = torch.load(best_ckpt, map_location=device)
        policy.load_state_dict(ckpt["model_state"])
        state_norm = ckpt["state_norm"]
        history = list(ckpt.get("history", history))
        logger.info("Loaded best RL checkpoint for final evaluation: %s", best_ckpt)
    return policy, state_norm, history


def load_best_or_latest_policy() -> Tuple[ActorCritic, Dict[str, Any]]:
    device = torch.device(CFG.torch_device)
    ckpt_path = CHECKPOINT_DIR / "policy_best.pt"
    if not ckpt_path.exists():
        ckpt_path = CHECKPOINT_DIR / "policy_latest.pt"
    if not ckpt_path.exists():
        raise FileNotFoundError("No RL checkpoint found. Run training first or set RUN_TRAIN_RL=1.")
    ckpt = torch.load(ckpt_path, map_location=device)
    state_dim = len(ckpt.get("feature_names", FEATURE_NAMES))
    n_actions = len(ckpt.get("action_space", ACTION_SPACE))
    policy = ActorCritic(state_dim=state_dim, n_actions=n_actions, hidden_dim=int(ckpt.get("config", {}).get("hidden_dim", CFG.hidden_dim))).to(device)
    policy.load_state_dict(ckpt["model_state"])
    logger.info("Loaded RL policy checkpoint: %s", ckpt_path)
    return policy, ckpt["state_norm"]


# ============================================================
# 7. Stress-test grouping and final tables
# ============================================================

def proxy_stress_groups(records: Sequence[Dict[str, Any]]) -> Dict[str, List[Dict[str, Any]]]:
    if not records:
        return {}
    states = compute_state_matrix(records)
    cols = {name: i for i, name in enumerate(FEATURE_NAMES)}
    gt_counts = np.asarray([int(r.get("gt_count", 0)) for r in records], dtype=np.float32)
    small = states[:, cols["small_box_ratio"]]
    overlap = states[:, cols["overlap_ratio_iou50"]]
    conf_mean = states[:, cols["conf_mean"]]

    q_gt = float(np.quantile(gt_counts, 0.75)) if len(gt_counts) else 0.0
    q_small = float(np.quantile(small, 0.75)) if len(small) else 0.0
    q_overlap = float(np.quantile(overlap, 0.75)) if len(overlap) else 0.0
    med_conf = float(np.median(conf_mean)) if len(conf_mean) else 0.0

    groups: Dict[str, List[Dict[str, Any]]] = {
        "heavy_occlusion_proxy_small_lowconf": [],
        "overlapping_fruit_clusters_proxy": [],
        "dense_count_proxy": [],
        "mixed_difficult_cases_proxy": [],
    }
    for i, r in enumerate(records):
        is_occ = bool(small[i] >= q_small and conf_mean[i] <= med_conf)
        is_ov = bool(overlap[i] >= q_overlap and overlap[i] > 0)
        is_dense = bool(gt_counts[i] >= q_gt and gt_counts[i] > 0)
        if is_occ:
            groups["heavy_occlusion_proxy_small_lowconf"].append(r)
        if is_ov:
            groups["overlapping_fruit_clusters_proxy"].append(r)
        if is_dense:
            groups["dense_count_proxy"].append(r)
        if is_occ or is_ov or is_dense:
            groups["mixed_difficult_cases_proxy"].append(r)
    return groups


def compare_methods(
    train_or_val_records: Sequence[Dict[str, Any]],
    test_records: Sequence[Dict[str, Any]],
    policy: ActorCritic,
    state_norm: Dict[str, Any],
) -> Tuple[pd.DataFrame, Dict[str, pd.DataFrame]]:
    per_image: Dict[str, pd.DataFrame] = {}
    comparison_rows: List[Dict[str, Any]] = []

    fixed_metrics, fixed_df = evaluate_fixed(test_records, CFG.fixed_conf, CFG.fixed_iou, CFG.fixed_scale)
    per_image["fixed_threshold"] = fixed_df
    comparison_rows.append(
        {
            "Method": "Fixed threshold",
            "Adaptive per image": "No",
            "conf": CFG.fixed_conf,
            "iou": CFG.fixed_iou,
            "scale": CFG.fixed_scale,
            **fixed_metrics,
        }
    )

    hard_best, hard_grid_df = tune_grid(train_or_val_records, method="hard")
    atomic_write_csv(EXP_DIR / "validation_grid_search_full.csv", hard_grid_df)
    if hard_best:
        grid_metrics, grid_df = evaluate_fixed(test_records, float(hard_best["conf"]), float(hard_best["iou"]), float(hard_best["scale"]))
        per_image["validation_grid_search"] = grid_df
        comparison_rows.append(
            {
                "Method": "Validation grid search",
                "Adaptive per image": "No",
                "conf": float(hard_best["conf"]),
                "iou": float(hard_best["iou"]),
                "scale": float(hard_best["scale"]),
                **grid_metrics,
            }
        )

    soft_best, soft_grid_df = tune_grid(train_or_val_records, method="soft")
    atomic_write_csv(EXP_DIR / "soft_nms_grid_search_full.csv", soft_grid_df)
    if soft_best:
        soft_metrics, soft_df = evaluate_soft(
            test_records,
            float(soft_best["conf"]),
            float(soft_best["iou"]),
            float(soft_best.get("sigma", 0.5)),
            float(soft_best["scale"]),
        )
        per_image["soft_nms"] = soft_df
        comparison_rows.append(
            {
                "Method": "Soft NMS",
                "Adaptive per image": "Partial",
                "conf": float(soft_best["conf"]),
                "iou": float(soft_best["iou"]),
                "sigma": float(soft_best.get("sigma", 0.5)),
                "scale": float(soft_best["scale"]),
                **soft_metrics,
            }
        )

    rl_metrics, rl_df = evaluate_policy(policy, test_records, state_norm, split_name=CFG.final_test_split, deterministic=True)
    per_image["rl_refinement"] = rl_df
    comparison_rows.append(
        {
            "Method": "RL refinement",
            "Adaptive per image": "Yes",
            "conf": "policy",
            "iou": "policy",
            "scale": "policy",
            **rl_metrics,
        }
    )

    comp = pd.DataFrame(comparison_rows)
    return comp, per_image


def stress_test_table(
    records: Sequence[Dict[str, Any]],
    best_grid: Optional[Dict[str, Any]],
    best_soft: Optional[Dict[str, Any]],
    policy: ActorCritic,
    state_norm: Dict[str, Any],
) -> pd.DataFrame:
    groups = proxy_stress_groups(records)
    rows = []
    for scenario, subset in groups.items():
        if not subset:
            continue
        fixed_metrics, _ = evaluate_fixed(subset, CFG.fixed_conf, CFG.fixed_iou, CFG.fixed_scale)
        if best_soft:
            soft_metrics, _ = evaluate_soft(subset, float(best_soft["conf"]), float(best_soft["iou"]), float(best_soft.get("sigma", 0.5)), float(best_soft["scale"]))
        else:
            soft_metrics = count_metrics([], [])
        if best_grid:
            grid_metrics, _ = evaluate_fixed(subset, float(best_grid["conf"]), float(best_grid["iou"]), float(best_grid["scale"]))
        else:
            grid_metrics = count_metrics([], [])
        rl_metrics, _ = evaluate_policy(policy, subset, state_norm, split_name=scenario, deterministic=True)
        rows.append(
            {
                "Stress scenario": scenario,
                "n_images": len(subset),
                "Fixed threshold MAE": fixed_metrics.get("MAE_count"),
                "Soft NMS MAE": soft_metrics.get("MAE_count"),
                "Grid search MAE": grid_metrics.get("MAE_count"),
                "RL refinement MAE": rl_metrics.get("MAE_count"),
                "Fixed threshold MAPE": fixed_metrics.get("MAPE_count"),
                "Soft NMS MAPE": soft_metrics.get("MAPE_count"),
                "Grid search MAPE": grid_metrics.get("MAPE_count"),
                "RL refinement MAPE": rl_metrics.get("MAPE_count"),
            }
        )
    return pd.DataFrame(rows)


def combine_split_records(cache_by_split: Dict[str, List[Dict[str, Any]]], split_spec: str) -> List[Dict[str, Any]]:
    out: List[Dict[str, Any]] = []
    for s in [x.strip() for x in split_spec.split(",") if x.strip()]:
        out.extend(cache_by_split.get(s, []))
    return out


def resolve_existing_split(cache_by_split: Dict[str, List[Dict[str, Any]]], preferred: str) -> str:
    for s in [preferred, "test", "val", "train"]:
        if cache_by_split.get(s):
            return s
    raise RuntimeError("No non-empty split cache found.")


# ============================================================
# 8. Main
# ============================================================

def main() -> None:
    started = time.time()
    detector_path = resolve_detector_path()
    custom_ok = register_custom_modules()

    run_config = {
        "project_root": str(PROJECT_ROOT),
        "data_dir": str(DATA_DIR),
        "data_yaml": str(DATA_YAML),
        "output_root": str(OUTPUT_ROOT),
        "exp_dir": str(EXP_DIR),
        "run_dir": str(RUN_DIR),
        "artifact_dir": str(ARTIFACT_DIR),
        "log_path": str(LOG_PATH),
        "local_log_path": str(LOCAL_LOG_PATH),
        "detector_path": str(detector_path),
        "custom_modules_registered": custom_ok,
        "class_names": CLASS_NAMES,
        "target_cls": TARGET_CLS,
        "target_class_name": TARGET_CLASS_NAME,
        "action_space_size": len(ACTION_SPACE),
        "feature_names": FEATURE_NAMES,
        "config": asdict(CFG),
        "platform": {
            "python": sys.version,
            "platform": platform.platform(),
            "torch": torch.__version__,
            "cuda_available": torch.cuda.is_available(),
            "cuda_device_count": torch.cuda.device_count() if torch.cuda.is_available() else 0,
        },
        "started_at": datetime.now().isoformat(timespec="seconds"),
    }
    atomic_write_json(EXP_DIR / "run_config.json", run_config)
    logger.info("Run config written: %s", EXP_DIR / "run_config.json")
    logger.info("DATA_DIR=%s", DATA_DIR)
    logger.info("OUTPUT_ROOT=%s", OUTPUT_ROOT)
    logger.info("DATA_YAML=%s", DATA_YAML)
    logger.info("DETECTOR=%s", detector_path)
    logger.info("Target class=%s (%s)", TARGET_CLS, TARGET_CLASS_NAME)
    logger.info("Action space size=%d", len(ACTION_SPACE))

    cache_by_split: Dict[str, List[Dict[str, Any]]] = {}
    for split in [x.strip() for x in CFG.cache_splits.split(",") if x.strip()]:
        cache_by_split[split] = build_or_load_candidate_cache(split, detector_path)
        logger.info("Split %s records=%d", split, len(cache_by_split[split]))

    split_summary = []
    for s, rows in cache_by_split.items():
        split_summary.append(
            {
                "split": s,
                "n_images": len(rows),
                "gt_total": int(sum(int(r.get("gt_count", 0)) for r in rows)),
                "candidate_total": int(sum(int(r.get("candidate_count", 0)) for r in rows)),
                "candidate_target_total": int(sum(int(r.get("candidate_target_count", 0)) for r in rows)),
                "avg_predict_ms": float(np.mean([float(r.get("predict_ms", 0.0)) for r in rows])) if rows else math.nan,
            }
        )
    atomic_write_csv(EXP_DIR / "candidate_cache_summary.csv", pd.DataFrame(split_summary))

    train_records = combine_split_records(cache_by_split, CFG.rl_train_splits)
    policy_val_split = resolve_existing_split(cache_by_split, CFG.policy_val_split)
    final_test_split = resolve_existing_split(cache_by_split, CFG.final_test_split)
    val_records = cache_by_split[policy_val_split]
    test_records = cache_by_split[final_test_split]
    logger.info("RL train splits=%s records=%d", CFG.rl_train_splits, len(train_records))
    logger.info("Policy validation split=%s records=%d", policy_val_split, len(val_records))
    logger.info("Final test split=%s records=%d", final_test_split, len(test_records))

    if CFG.run_train_rl:
        policy, state_norm, history = train_rl_policy(train_records, val_records, detector_path)
    else:
        policy, state_norm = load_best_or_latest_policy()
        history = []

    if CFG.run_final_eval:
        comp, per_image = compare_methods(train_records if train_records else val_records, test_records, policy, state_norm)
        table_441 = EXP_DIR / "table_4_4_1_postprocess_comparison.csv"
        atomic_write_csv(table_441, comp)
        for name, df in per_image.items():
            atomic_write_csv(EXP_DIR / f"per_image_{final_test_split}_{name}.csv", df)

        hard_best, hard_grid_df = tune_grid(train_records if train_records else val_records, method="hard")
        soft_best, soft_grid_df = tune_grid(train_records if train_records else val_records, method="soft")
        stress = stress_test_table(test_records, hard_best, soft_best, policy, state_norm)
        table_442 = EXP_DIR / "table_4_4_2_stress_test.csv"
        atomic_write_csv(table_442, stress)

        # Deployment/runtime artifact: policy checkpoint + JSON config for lightweight inference.
        runtime_config = {
            "policy_checkpoint": str((CHECKPOINT_DIR / "policy_best.pt") if (CHECKPOINT_DIR / "policy_best.pt").exists() else (CHECKPOINT_DIR / "policy_latest.pt")),
            "detector_path": str(detector_path),
            "data_yaml": str(DATA_YAML),
            "target_cls": TARGET_CLS,
            "target_class_name": TARGET_CLASS_NAME,
            "class_names": CLASS_NAMES,
            "feature_names": FEATURE_NAMES,
            "state_norm": state_norm,
            "action_space": ACTION_SPACE,
            "candidate_conf_for_runtime": CFG.cand_conf,
            "candidate_iou_for_runtime": CFG.cand_iou,
            "img_size": CFG.img_size,
            "created_at": datetime.now().isoformat(timespec="seconds"),
        }
        runtime_path = ARTIFACT_DIR / f"rl_policy_{DATASET_NAME}_runtime_config.json"
        atomic_write_json(runtime_path, runtime_config)

        logger.info("Final comparison table: %s", table_441)
        logger.info("Stress-test table: %s", table_442)
        logger.info("Runtime RL config: %s", runtime_path)
        print("\n=== Table 4.4.1: Post-processing comparison ===")
        print(comp.to_string(index=False))
        if not stress.empty:
            print("\n=== Table 4.4.2: Stress-test proxy groups ===")
            print(stress.to_string(index=False))

    atomic_write_json(
        EXP_DIR / "run_status.json",
        {
            "status": "complete",
            "elapsed_sec": time.time() - started,
            "completed_at": datetime.now().isoformat(timespec="seconds"),
            "exp_dir": str(EXP_DIR),
            "run_dir": str(RUN_DIR),
            "checkpoint_dir": str(CHECKPOINT_DIR),
            "artifact_dir": str(ARTIFACT_DIR),
            "log_path": str(LOG_PATH),
        },
    )
    logger.info("[07] Done -> %s", EXP_DIR)
    print("\n[07] Done")
    print("EXP_DIR:", EXP_DIR)
    print("CHECKPOINT_DIR:", CHECKPOINT_DIR)
    print("ARTIFACT_DIR:", ARTIFACT_DIR)
    print("LOG_PATH:", LOG_PATH)


if __name__ == "__main__":
    try:
        main()
    except Exception as e:
        logger.exception("Fatal error in 07_rl_count_refine.py")
        atomic_write_json(
            EXP_DIR / "run_status.json",
            {
                "status": "failed",
                "error": repr(e),
                "traceback": traceback.format_exc(),
                "failed_at": datetime.now().isoformat(timespec="seconds"),
                "log_path": str(LOG_PATH),
            },
        )
        raise


/home/diy-hus/miniconda3/envs/fruitnet/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


2026-06-30 18:47:42,795 | INFO | Registered FruitNet custom modules for Ultralytics.
2026-06-30 18:47:42,842 | INFO | Run config written: /mnt/f/fruitnet-chili-yield-outputs/results/exp03_rl_finetune_data/run_config.json
2026-06-30 18:47:42,844 | INFO | DATA_DIR=/mnt/f/fruitnet-chili-yield-data/finetune_data
2026-06-30 18:47:42,844 | INFO | OUTPUT_ROOT=/mnt/f/fruitnet-chili-yield-outputs
2026-06-30 18:47:42,845 | INFO | DATA_YAML=/home/diy-hus/fruitnet-chili-yield/configs/data/finetune_data.yaml
2026-06-30 18:47:42,846 | INFO | DETECTOR=/mnt/f/fruitnet-chili-yield-outputs/artifacts/finetuned/fruitnet_gcs_finetune_data_best.pt
2026-06-30 18:47:42,847 | INFO | Target class=0 (chili)
2026-06-30 18:47:42,849 | INFO | Action space size=490
2026-06-30 18:52:07,242 | INFO | Building candidate cache split=train images=10924 detector=/mnt/f/fruitnet-chili-yield-outputs/artifacts/finetuned/fruitnet_gcs_finetune_data_best.pt conf=0.0010 iou=0.99 max_det=1000


cache:train: 100%|██████████| 10924/10924 [11:33<00:00, 15.76it/s]


2026-06-30 19:03:43,608 | INFO | Candidate cache complete split=train -> /mnt/f/fruitnet-chili-yield-outputs/results/exp03_rl_finetune_data/candidate_cache/train_candidates.jsonl
2026-06-30 19:03:43,610 | INFO | Split train records=10924
2026-06-30 19:03:57,558 | INFO | Building candidate cache split=val images=569 detector=/mnt/f/fruitnet-chili-yield-outputs/artifacts/finetuned/fruitnet_gcs_finetune_data_best.pt conf=0.0010 iou=0.99 max_det=1000


cache:val: 100%|██████████| 569/569 [00:18<00:00, 30.74it/s] 


2026-06-30 19:04:16,428 | INFO | Candidate cache complete split=val -> /mnt/f/fruitnet-chili-yield-outputs/results/exp03_rl_finetune_data/candidate_cache/val_candidates.jsonl
2026-06-30 19:04:16,430 | INFO | Split val records=569
2026-06-30 19:04:19,086 | INFO | Building candidate cache split=test images=115 detector=/mnt/f/fruitnet-chili-yield-outputs/artifacts/finetuned/fruitnet_gcs_finetune_data_best.pt conf=0.0010 iou=0.99 max_det=1000


cache:test: 100%|██████████| 115/115 [00:03<00:00, 29.35it/s]

2026-06-30 19:04:23,248 | INFO | Candidate cache complete split=test -> /mnt/f/fruitnet-chili-yield-outputs/results/exp03_rl_finetune_data/candidate_cache/test_candidates.jsonl


2026-06-30 19:04:23,250 | INFO | Split test records=115
2026-06-30 19:04:23,283 | INFO | RL train splits=val records=569
2026-06-30 19:04:23,284 | INFO | Policy validation split=val records=569
2026-06-30 19:04:23,285 | INFO | Final test split=test records=115
2026-06-30 19:04:24,122 | INFO | RL train start: records=569 val_records=569 start_epoch=0 epochs=300 actions=490 state_dim=23 device=cuda
2026-06-30 19:04:26,667 | INFO | New best RL policy at epoch=1 val_MAE=0.4938 -> /mnt/f/fruitnet-chili-yield-outputs/runs/rl/finetune_data/checkpoints/policy_best.pt
2026-06-30 19:04:26,706 | INFO | RL epoch 001/300 | reward=-0.3210 | loss=0.2800 | val_MAE=0.4938 | best_MAE=0.4938 | 2.6s
2026-06-30 19:04:27,138 | INFO | RL epoch 002/300 | reward=-0.3069 | loss=0.2588 | val_MAE=NA | best_MAE=0.4938 | 0.4s
2026-06-30 19:04:27,559 | INFO | RL epoch 003/300 | reward=-0.2814 | loss=0.1654 | val_MAE=NA | best_MAE=0.4938 | 0.4s
2026-06-30 19:04:27,961 | INFO | RL epoch 004/300 | reward=-0.2676 | loss